# ResNet18 Plastic Classifier - Terminal Runner
## Load Trained Model and Run Predictions

In [1]:
# Install required packages (if needed)
# !pip install torch torchvision pillow flask flask-cors

In [2]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import os
import sys

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Device: {device}')
print(f'📁 Current Directory: {os.getcwd()}')

🖥️  Device: cpu
📁 Current Directory: c:\Users\vimal\OneDrive\Desktop\FINAL AIML


## 1. Load Trained ResNet18 Model

In [3]:
MODEL_PATH = 'best_model_ResNet18.pth'

if not os.path.exists(MODEL_PATH):
    print(f'❌ Model file not found: {MODEL_PATH}')
    print('Please ensure the model file is in the current directory.')
    sys.exit(1)

# Load checkpoint
checkpoint = torch.load(MODEL_PATH, map_location=device)
num_classes = checkpoint['num_classes']
idx_to_class = checkpoint['idx_to_class']
plastic_types = checkpoint['plastic_types']
best_acc = checkpoint['best_acc']

print(f'✅ Model loaded successfully!')
print(f'📊 Classes: {plastic_types}')
print(f'🎯 Number of classes: {num_classes}')
print(f'🏆 Best Validation Accuracy: {best_acc:.4f} ({best_acc*100:.2f}%)')

✅ Model loaded successfully!
📊 Classes: ['HDPE', 'LDPE', 'OTHERS', 'PET', 'PP', 'PS']
🎯 Number of classes: 6
🏆 Best Validation Accuracy: 0.9953 (99.53%)


In [4]:
# Create model architecture
model = models.resnet18(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

print('✅ Model architecture loaded and set to evaluation mode')

c:\Users\vimal\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\vimal\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


✅ Model architecture loaded and set to evaluation mode


## 2. Define Image Preprocessing

In [5]:
# Image transformation pipeline
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print('✅ Image preprocessing pipeline ready')

✅ Image preprocessing pipeline ready


## 3. Plastic Information Database

In [6]:
plastic_db = {
    'PET': {
        'full_name': 'Polyethylene Terephthalate',
        'recycling_code': '#1 PET',
        'recyclability': 'Highly Recyclable',
        'color': 'green',
        'common_uses': 'Water bottles, soft drink bottles, food containers',
        'recycling_tips': 'Rinse and remove caps before recycling'
    },
    'HDPE': {
        'full_name': 'High-Density Polyethylene',
        'recycling_code': '#2 HDPE',
        'recyclability': 'Highly Recyclable',
        'color': 'green',
        'common_uses': 'Milk jugs, detergent bottles, shampoo bottles',
        'recycling_tips': 'Clean and dry before recycling'
    },
    'PVC': {
        'full_name': 'Polyvinyl Chloride',
        'recycling_code': '#3 PVC',
        'recyclability': 'Rarely Recyclable',
        'color': 'red',
        'common_uses': 'Pipes, vinyl siding, credit cards',
        'recycling_tips': 'Check with local facilities, often not accepted'
    },
    'LDPE': {
        'full_name': 'Low-Density Polyethylene',
        'recycling_code': '#4 LDPE',
        'recyclability': 'Sometimes Recyclable',
        'color': 'orange',
        'common_uses': 'Plastic bags, squeeze bottles, bread bags',
        'recycling_tips': 'Many stores accept plastic bags for recycling'
    },
    'PP': {
        'full_name': 'Polypropylene',
        'recycling_code': '#5 PP',
        'recyclability': 'Recyclable',
        'color': 'blue',
        'common_uses': 'Yogurt containers, bottle caps, straws',
        'recycling_tips': 'Clean containers before recycling'
    },
    'PS': {
        'full_name': 'Polystyrene',
        'recycling_code': '#6 PS',
        'recyclability': 'Rarely Recyclable',
        'color': 'red',
        'common_uses': 'Foam cups, takeout containers, packing peanuts',
        'recycling_tips': 'Most facilities do not accept, check locally'
    },
    'OTHERS': {
        'full_name': 'Other Plastics',
        'recycling_code': '#7 OTHER',
        'recyclability': 'Check Guidelines',
        'color': 'gray',
        'common_uses': 'Mixed plastics, polycarbonate, bioplastics',
        'recycling_tips': 'Varies by type, check with local recycling center'
    }
}

print('✅ Plastic information database loaded')

✅ Plastic information database loaded


## 4. Prediction Function

In [7]:
def predict_plastic(image_path):
    """
    Predict plastic type from image
    
    Args:
        image_path: Path to image file
    
    Returns:
        dict: Prediction results with plastic type, confidence, and info
    """
    try:
        # Load and preprocess image
        image = Image.open(image_path).convert('RGB')
        img_tensor = transform(image).unsqueeze(0).to(device)
        
        # Make prediction
        with torch.no_grad():
            outputs = model(img_tensor)
            probs = torch.nn.functional.softmax(outputs, dim=1)
            confidence, predicted = torch.max(probs, 1)
        
        # Convert idx_to_class keys to integers if they're strings
        if isinstance(list(idx_to_class.keys())[0], str):
            idx_to_class_int = {int(k): v for k, v in idx_to_class.items()}
        else:
            idx_to_class_int = idx_to_class
        
        predicted_idx = predicted.item()
        plastic_type = idx_to_class_int.get(predicted_idx, 'UNKNOWN')
        confidence_score = confidence.item()
        
        # Get all class probabilities
        all_probs = probs[0].cpu().numpy()
        
        return {
            'success': True,
            'plastic_type': plastic_type,
            'confidence': confidence_score,
            'info': plastic_db.get(plastic_type, {}),
            'all_probabilities': {idx_to_class_int[i]: float(all_probs[i]) for i in range(len(all_probs))}
        }
    except Exception as e:
        return {
            'success': False,
            'error': str(e)
        }

print('✅ Prediction function defined')

✅ Prediction function defined


## 5. Display Results Function

In [8]:
def display_results(result):
    """
    Display prediction results in a formatted way
    """
    if not result['success']:
        print(f"\n❌ Error: {result['error']}")
        return
    
    plastic_type = result['plastic_type']
    confidence = result['confidence']
    info = result['info']
    
    print('\n' + '='*70)
    print('🔍 PLASTIC CLASSIFICATION RESULTS')
    print('='*70)
    
    # Main prediction
    print(f"\n🎯 Predicted Type: {plastic_type}")
    print(f"📊 Confidence: {confidence*100:.2f}%")
    
    if info:
        print(f"\n📝 Full Name: {info['full_name']}")
        print(f"♻️  Recycling Code: {info['recycling_code']}")
        print(f"🌍 Recyclability: {info['recyclability']}")
        print(f"📦 Common Uses: {info['common_uses']}")
        print(f"💡 Recycling Tips: {info['recycling_tips']}")
    
    # All probabilities
    print(f"\n📈 All Class Probabilities:")
    for cls, prob in sorted(result['all_probabilities'].items(), key=lambda x: x[1], reverse=True):
        bar = '█' * int(prob * 50)
        print(f"  {cls:8s}: {prob*100:5.2f}% {bar}")
    
    print('\n' + '='*70)

print('✅ Display function defined')

✅ Display function defined


## 6. Test with Sample Images

In [9]:
# Test with images from test directory
TEST_DIR = r'c:\Users\vimal\OneDrive\Desktop\FINAL AIML\Waste segregation.v1i.multiclass\test'

if os.path.exists(TEST_DIR):
    # Get sample images
    test_images = [f for f in os.listdir(TEST_DIR) if f.endswith(('.jpg', '.jpeg', '.png'))]
    
    if test_images:
        print(f"\n📁 Found {len(test_images)} test images")
        print("\nTesting with first 3 images...\n")
        
        for img_file in test_images[:3]:
            img_path = os.path.join(TEST_DIR, img_file)
            print(f"\n🖼️  Testing image: {img_file}")
            result = predict_plastic(img_path)
            display_results(result)
    else:
        print("⚠️  No test images found")
else:
    print(f"⚠️  Test directory not found: {TEST_DIR}")


📁 Found 1033 test images

Testing with first 3 images...


🖼️  Testing image: 100Mesa-de-trabajo-1_png_jpg.rf.4f218b78779f8053e80a51b409e42b08.jpg

🔍 PLASTIC CLASSIFICATION RESULTS

🎯 Predicted Type: PP
📊 Confidence: 87.98%

📝 Full Name: Polypropylene
♻️  Recycling Code: #5 PP
🌍 Recyclability: Recyclable
📦 Common Uses: Yogurt containers, bottle caps, straws
💡 Recycling Tips: Clean containers before recycling

📈 All Class Probabilities:
  PP      : 87.98% ███████████████████████████████████████████
  PET     :  7.78% ███
  HDPE    :  2.64% █
  OTHERS  :  1.25% 
  PS      :  0.34% 
  LDPE    :  0.01% 


🖼️  Testing image: 114Mesa-de-trabajo-1_png_jpg.rf.daf0723fcb6b2058f0672e9e87afc7d2.jpg

🔍 PLASTIC CLASSIFICATION RESULTS

🎯 Predicted Type: PP
📊 Confidence: 89.82%

📝 Full Name: Polypropylene
♻️  Recycling Code: #5 PP
🌍 Recyclability: Recyclable
📦 Common Uses: Yogurt containers, bottle caps, straws
💡 Recycling Tips: Clean containers before recycling

📈 All Class Probabilities:
  PP     

## 7. Interactive Prediction (Custom Image Path)

In [10]:
# Predict on a custom image
# Replace with your image path
custom_image_path = r'c:\Users\vimal\OneDrive\Desktop\FINAL AIML\Waste segregation.v1i.multiclass\test\PET_1.jpg'

if os.path.exists(custom_image_path):
    print(f"\n🖼️  Analyzing: {custom_image_path}")
    result = predict_plastic(custom_image_path)
    display_results(result)
else:
    print(f"⚠️  Image not found: {custom_image_path}")
    print("\nTo test with your own image:")
    print("1. Update the 'custom_image_path' variable above")
    print("2. Run this cell again")

⚠️  Image not found: c:\Users\vimal\OneDrive\Desktop\FINAL AIML\Waste segregation.v1i.multiclass\test\PET_1.jpg

To test with your own image:
1. Update the 'custom_image_path' variable above
2. Run this cell again


## 8. Batch Prediction on Multiple Images

In [11]:
def batch_predict(image_folder, max_images=10):
    """
    Predict plastic types for multiple images in a folder
    """
    if not os.path.exists(image_folder):
        print(f"❌ Folder not found: {image_folder}")
        return
    
    image_files = [f for f in os.listdir(image_folder) if f.endswith(('.jpg', '.jpeg', '.png'))]
    
    if not image_files:
        print("❌ No images found in folder")
        return
    
    print(f"\n📁 Processing {min(len(image_files), max_images)} images...\n")
    
    results_summary = {}
    
    for i, img_file in enumerate(image_files[:max_images], 1):
        img_path = os.path.join(image_folder, img_file)
        result = predict_plastic(img_path)
        
        if result['success']:
            plastic_type = result['plastic_type']
            confidence = result['confidence']
            print(f"{i}. {img_file:30s} → {plastic_type:8s} ({confidence*100:.1f}%)")
            
            if plastic_type not in results_summary:
                results_summary[plastic_type] = 0
            results_summary[plastic_type] += 1
    
    # Summary
    print("\n" + "="*70)
    print("📊 BATCH PREDICTION SUMMARY")
    print("="*70)
    for plastic_type, count in sorted(results_summary.items(), key=lambda x: x[1], reverse=True):
        print(f"  {plastic_type:8s}: {count} images")
    print("="*70)

# Run batch prediction
batch_predict(TEST_DIR, max_images=10)


📁 Processing 10 images...

1. 100Mesa-de-trabajo-1_png_jpg.rf.4f218b78779f8053e80a51b409e42b08.jpg → PP       (88.0%)
2. 114Mesa-de-trabajo-1_png_jpg.rf.daf0723fcb6b2058f0672e9e87afc7d2.jpg → PP       (89.8%)
3. 118Mesa-de-trabajo-1_png_jpg.rf.acf89c0dbf0a88b2895d62cda37d8a87.jpg → PP       (42.1%)
4. 121Mesa-de-trabajo-1_png_jpg.rf.db118cf2e213d0f5ebaeeb4b3b6d3136.jpg → PP       (80.5%)
5. 123Mesa-de-trabajo-1_png_jpg.rf.0b2ddfc9e00c2211b9758906c1315eb9.jpg → PP       (59.7%)
6. 124Mesa-de-trabajo-1_png_jpg.rf.56dc250aebe8f2388c39cda8cd47a3a5.jpg → PP       (59.0%)
7. 126Mesa-de-trabajo-1_png_jpg.rf.1a104a843a93f5702644ab3b6232da5b.jpg → PP       (58.5%)
8. 130Mesa-de-trabajo-1_png_jpg.rf.7b850d2b34e770ad37cf64e74cc75107.jpg → PET      (51.9%)
9. 156Mesa-de-trabajo-1_png_jpg.rf.8d7e70f7947a1a3c66ef1c6cb8b0422f.jpg → PP       (73.8%)
10. 15Mesa-de-trabajo-1_png_jpg.rf.ef2d293e4ac6f7d0973fc7bcc2064ebd.jpg → PP       (75.1%)

📊 BATCH PREDICTION SUMMARY
  PP      : 9 images
  PET     : 1

## 9. Start Flask Web Server (Optional)

In [12]:
# Check if Flask app exists
if os.path.exists('app.py'):
    print("✅ Flask app.py found!")
    print("\nTo start the web server, run in terminal:")
    print("  python app.py")
    print("\nThen open browser: http://localhost:5000")
else:
    print("⚠️  app.py not found. Run 03_resnet18_deployment.ipynb to create it.")

✅ Flask app.py found!

To start the web server, run in terminal:
  python app.py

Then open browser: http://localhost:5000


## 10. Model Information Summary

In [13]:
print('\n' + '='*70)
print('📋 MODEL INFORMATION SUMMARY')
print('='*70)
print(f"Model Architecture: ResNet18")
print(f"Number of Classes: {num_classes}")
print(f"Classes: {', '.join(plastic_types)}")
print(f"Best Validation Accuracy: {best_acc:.4f} ({best_acc*100:.2f}%)")
print(f"Input Image Size: 224x224 pixels")
print(f"Device: {device}")
print(f"Model File: {MODEL_PATH}")
print('='*70)

print("\n✅ Model is ready for predictions!")
print("\n📝 Usage:")
print("  1. Use predict_plastic(image_path) for single predictions")
print("  2. Use batch_predict(folder_path) for multiple images")
print("  3. Run 'python app.py' to start web interface")


📋 MODEL INFORMATION SUMMARY
Model Architecture: ResNet18
Number of Classes: 6
Classes: HDPE, LDPE, OTHERS, PET, PP, PS
Best Validation Accuracy: 0.9953 (99.53%)
Input Image Size: 224x224 pixels
Device: cpu
Model File: best_model_ResNet18.pth

✅ Model is ready for predictions!

📝 Usage:
  1. Use predict_plastic(image_path) for single predictions
  2. Use batch_predict(folder_path) for multiple images
  3. Run 'python app.py' to start web interface


## 11. 🎯 DRAG & DROP YOUR IMAGE PATH HERE

In [14]:
# =============================================================================
# 🎯 INTERACTIVE IMAGE PATH INPUT
# =============================================================================
# This cell will prompt you to enter the image path
# Just run the cell and paste your path when asked!
# =============================================================================

print('='*70)
print('🎯 PLASTIC WASTE CLASSIFIER - IMAGE INPUT')
print('='*70)
print('\n📝 INSTRUCTIONS:')
print('   1. Run this cell')
print('   2. When prompted, paste your image path')
print('   3. Press Enter to get prediction!')
print('\n💡 EXAMPLES:')
print('   C:/Users/vimal/Desktop/bottle.jpg')
print('   test/image.jpg')
print('   Or drag and drop your file path here')
print('='*70)

# Get input from user
image_path = input('\n👉 Enter image path (or folder path for batch): ').strip()

# =============================================================================
# AUTOMATIC PREDICTION - NO NEED TO EDIT BELOW
# =============================================================================

if image_path:
    # Clean up the path (remove quotes if any)
    image_path = image_path.strip('"').strip("'")
    
    print('\n' + '='*70)
    print('🚀 STARTING PREDICTION...')
    print('='*70)
    print(f'📁 Path: {image_path}')
    
    # Check if file exists
    if os.path.exists(image_path):
        # Check if it's a file or folder
        if os.path.isfile(image_path):
            # Single image prediction
            print(f'✅ File found!')
            print(f'📏 File size: {os.path.getsize(image_path) / 1024:.2f} KB')
            
            result = predict_plastic(image_path)
            display_results(result)
            
        elif os.path.isdir(image_path):
            # Folder - batch prediction
            print(f'✅ Folder found!')
            print(f'🔄 Running batch prediction...\n')
            
            batch_predict(image_path, max_images=20)
    else:
        print('\n❌ ERROR: Path not found!')
        print(f'   Path: {image_path}')
        print('\n💡 TIPS:')
        print('   1. Make sure the path is correct')
        print('   2. Remove any extra quotes')
        print('   3. Check for typos in the path')
        print('   4. Try using forward slashes: C:/path/to/file.jpg')
        print('='*70)
else:
    print('\n⚠️  No path provided. Please run the cell again and enter a path.')
    print('='*70)

🎯 PLASTIC WASTE CLASSIFIER - IMAGE INPUT

📝 INSTRUCTIONS:
   1. Run this cell
   2. When prompted, paste your image path
   3. Press Enter to get prediction!

💡 EXAMPLES:
   C:/Users/vimal/Desktop/bottle.jpg
   test/image.jpg
   Or drag and drop your file path here

🚀 STARTING PREDICTION...
📁 Path: C:\Users\vimal\OneDrive\Desktop\FINAL AIML\Waste segregation.v1i.multiclass\test\HDPE-74_jpg.rf.00ad2d28affe9f57d107501bd02976eb.jpg
✅ File found!
📏 File size: 19.40 KB

🔍 PLASTIC CLASSIFICATION RESULTS

🎯 Predicted Type: HDPE
📊 Confidence: 100.00%

📝 Full Name: High-Density Polyethylene
♻️  Recycling Code: #2 HDPE
🌍 Recyclability: Highly Recyclable
📦 Common Uses: Milk jugs, detergent bottles, shampoo bottles
💡 Recycling Tips: Clean and dry before recycling

📈 All Class Probabilities:
  HDPE    : 100.00% █████████████████████████████████████████████████
  PP      :  0.00% 
  LDPE    :  0.00% 
  OTHERS  :  0.00% 
  PS      :  0.00% 
  PET     :  0.00% 



## 12. 📋 QUICK TEST - Multiple Images at Once

In [15]:
# =============================================================================
# 📋 BATCH TEST - MULTIPLE IMAGES WITH INPUT
# =============================================================================
# Test multiple images by entering paths one by one
# =============================================================================

print('='*70)
print('📋 BATCH IMAGE TESTING')
print('='*70)
print('\n📝 INSTRUCTIONS:')
print('   1. Enter how many images you want to test')
print('   2. Paste each image path when prompted')
print('   3. Get summary of all predictions!')
print('='*70)

# Get number of images
try:
    num_images = int(input('\n👉 How many images do you want to test? '))
    
    if num_images <= 0:
        print('⚠️  Please enter a positive number!')
    else:
        test_images_list = []
        
        print(f'\n📸 Enter {num_images} image paths:')
        for i in range(num_images):
            path = input(f'  Image {i+1}: ').strip().strip('"').strip("'")
            if path:
                test_images_list.append(path)
        
        # =============================================================================
        # AUTOMATIC PREDICTION
        # =============================================================================
        
        if test_images_list:
            print('\n' + '='*70)
            print(f'🚀 TESTING {len(test_images_list)} IMAGES')
            print('='*70)
            
            results_summary = {}
            successful = 0
            failed = 0
            
            for idx, img_path in enumerate(test_images_list, 1):
                print(f'\n[{idx}/{len(test_images_list)}] Testing: {os.path.basename(img_path)}')
                
                if os.path.exists(img_path):
                    result = predict_plastic(img_path)
                    
                    if result['success']:
                        plastic_type = result['plastic_type']
                        confidence = result['confidence']
                        print(f'     ✅ {plastic_type} ({confidence*100:.1f}%)')
                        
                        if plastic_type not in results_summary:
                            results_summary[plastic_type] = 0
                        results_summary[plastic_type] += 1
                        successful += 1
                    else:
                        print(f'     ❌ Error: {result["error"]}')
                        failed += 1
                else:
                    print(f'     ❌ File not found!')
                    failed += 1
            
            # Final Summary
            print('\n' + '='*70)
            print('📊 FINAL SUMMARY')
            print('='*70)
            print(f'Total Images: {len(test_images_list)}')
            print(f'Successful: {successful}')
            print(f'Failed: {failed}')
            
            if results_summary:
                print('\n🎯 Classification Results:')
                for plastic_type, count in sorted(results_summary.items(), key=lambda x: x[1], reverse=True):
                    percentage = (count / successful) * 100 if successful > 0 else 0
                    print(f'  {plastic_type:8s}: {count:2d} images ({percentage:5.1f}%)')
            
            print('='*70)
        else:
            print('\n⚠️  No valid images provided!')
            
except ValueError:
    print('\n❌ Invalid input! Please enter a number.')
except KeyboardInterrupt:
    print('\n\n⚠️  Operation cancelled by user.')

📋 BATCH IMAGE TESTING

📝 INSTRUCTIONS:
   1. Enter how many images you want to test
   2. Paste each image path when prompted
   3. Get summary of all predictions!

📸 Enter 1 image paths:

🚀 TESTING 1 IMAGES

[1/1] Testing: HDPE-74_jpg.rf.00ad2d28affe9f57d107501bd02976eb.jpg
     ✅ HDPE (100.0%)

📊 FINAL SUMMARY
Total Images: 1
Successful: 1
Failed: 0

🎯 Classification Results:
  HDPE    :  1 images (100.0%)
